In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

c:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.sample(5)

,id,dialogue,summary
2195,13682537,Luna: need to lost some weight!\r\nLuna: I can...,Luna wants to lose some weight because she can...
14693,13812564,Maria: Did you note down the lecture?\r\nJesus...,Professor was speaking too fast to take notes....
9582,13728983,Caro: hello it's such a long time i haven't se...,Sybille is back from New Orleans and is seeing...
1715,13728642,Claudio: Is the Nescafe coffee thing any good?...,Agnes likes Nescafe coffee machine. Claudio wa...
12990,13682225,"Eli: hey, I'm at the store. do you want anythi...",Eli is about to check out in a shop. Frank wan...


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
# random sampling = train data(4000 samples), and val data(500 samples)
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [7]:
train_data.shape

(4000, 3)

### Data Preprocessing

In [8]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # remove next lines \r\n
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <h1> </p>
    text = text.strip().lower()
    return text

In [9]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

### Tokenize

In [10]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [11]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    input = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)
    
    input["labels"] = targets["input_ids"] # token_ids => add to input as labels
    return input

In [12]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [13]:

train_dataset[0] 
# inputs ids - dialogue => token ids      (1 = EOS(end of sequence), then all 0 until attention mask => padding) => total tokens = max_length=512
# attention mask   =>   (1= for valid inputs, 0= for padded tokens)
# labels - target => summary token ids

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [14]:
len(train_dataset[0]["input_ids"])

512

### Working with our Model (T5)

In [15]:
# NLP = generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [16]:
# fine-tune
import torch

if torch.backends.mps.is_available(): # for mac
    device = torch.device("mps")
elif torch.cuda.is_available(): # for gpu
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
# Training Arguments

train_args = TrainingArguments(
    output_dir= "./results",  # save the model parameters
    
    num_train_epochs=6,
    weight_decay=0.01, # prevents the model from making weights too large so it generalizes better(otherwise it starts memorizing,instead of learning)
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    eval_strategy="epoch", # after every epoch, the model is validated
    save_strategy="epoch", # save after every epoch
    
    warmup_steps=500, # learning rate: 0 => 5e-05 (default lr) in 500 steps => slow learning at first (500 steps each epoch)
    fp16=True
)

In [18]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

### Train the Model

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.693000,0.380094
2,0.398000,0.359470
3,0.372600,0.354473
4,0.362600,0.349735
5,0.355500,0.349316
6,0.352200,0.348607


TrainOutput(global_step=3000, training_loss=0.9223082071940104, metrics={'train_runtime': 1327.436, 'train_samples_per_second': 18.08, 'train_steps_per_second': 2.26, 'total_flos': 3248203235328000.0, 'train_loss': 0.9223082071940104, 'epoch': 6.0})

### Save and Load the Model

In [20]:
# model load => fine-tune => save model

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\special_tokens_map.json',
 './saved_summary_model\\spiece.model',
 './saved_summary_model\\added_tokens.json')

In [21]:
# Load the saved model
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

### Test the core logic for summarization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # preprocess
    
    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt" # tokenizer returns pytorch tensors
    )
    
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # generate the summary => token ids will be generated
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,   # maximum 150 tokens for our summary
        num_beams = 4,    # returns best output from 4 generated outputs
        early_stopping = True   # model stops as soon as 4 outputs (num_beams=4) are generated
    )
    
    # convert token ids to text => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # special tokens = EOS, SEP
    return summary

In [33]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary:", summary)

Summary: ai adoption has significantly increased over the past few years. ai systems are becoming more capable due to advances in deep learning and access to large datasets. they can now perform complex tasks such as language understanding, image recognition, and even code generation.
